In [7]:
from pathlib import Path
import pandas as pd

# ============================================================
# SP500 GEMINI KLASÖRÜ
# ============================================================

sp500_dir = Path(
    r"D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis"
    r"\db\annotations\gemini\sp_500"
)

# Daha önce oluşturduğumuz çıktı dosyasını tekrar giriş olarak okuma
output_name = "SP500_gemini_all_clean.csv"

all_dfs = []

# ============================================================
# 1) TÜM CSV DOSYALARINI OKU
# ============================================================

for file in sorted(sp500_dir.glob("*.csv")):

    if file.name == output_name:
        continue

    try:
        df = pd.read_csv(
            file,
            encoding="utf-8-sig",
            dtype=str
        )

        df["source_file"] = file.name
        all_dfs.append(df)

        print(f"CSV  : {file.name:45} -> {len(df):4} kayıt")

    except Exception as e:
        print(f"CSV HATA: {file.name} -> {e}")

# ============================================================
# 2) TÜM TXT DOSYALARINI OKU
#    Bunlar TAB/TSV formatında
# ============================================================

for file in sorted(sp500_dir.glob("*.txt")):

    if file.name == "000_prompt.txt":
        continue

    try:
        df = pd.read_csv(
            file,
            sep="\t",
            encoding="utf-8-sig",
            dtype=str
        )

        df["source_file"] = file.name
        all_dfs.append(df)

        print(f"TXT  : {file.name:45} -> {len(df):4} kayıt")

    except Exception as e:
        print(f"TXT HATA: {file.name} -> {e}")

# ============================================================
# 3) HER ŞEYİ TEK DATAFRAME'DE BİRLEŞTİR
# ============================================================

sp500_gemini_df = pd.concat(
    all_dfs,
    ignore_index=True,
    sort=False
)

print("\n======================================")
print("HAM TOPLAM:", len(sp500_gemini_df))
print("======================================")

# ============================================================
# 4) annotation_id TEMİZLE
# ============================================================

sp500_gemini_df["annotation_id"] = (
    sp500_gemini_df["annotation_id"]
    .astype(str)
    .str.strip()
)

# ============================================================
# 5) SADECE SP500 ID'LERİ
# ============================================================

sp500_gemini_df = sp500_gemini_df[
    sp500_gemini_df["annotation_id"].str.match(
        r"^SP500_ANN_\d{4}$",
        na=False
    )
].copy()

# ============================================================
# 6) DUPLICATE TEMİZLE
# ============================================================

before = len(sp500_gemini_df)

sp500_gemini_df = (
    sp500_gemini_df
    .drop_duplicates(
        subset="annotation_id",
        keep="first"
    )
    .reset_index(drop=True)
)

print("Duplicate silinen:", before - len(sp500_gemini_df))
print("Unique annotation:", len(sp500_gemini_df))

# ============================================================
# 7) KOLONLARI DÜZENLE
# ============================================================

wanted_columns = [
    "annotation_id",
    "sample_id",
    "date",
    "finbert_label",
    "finbert_confidence",
    "TEXT",
    "gemini_label",
    "source_file"
]

sp500_gemini_df = sp500_gemini_df[
    [c for c in wanted_columns if c in sp500_gemini_df.columns]
]

# ============================================================
# 8) ID SIRALA
# ============================================================

sp500_gemini_df["id_num"] = (
    sp500_gemini_df["annotation_id"]
    .str.extract(r"(\d+)$")[0]
    .astype(int)
)

sp500_gemini_df = (
    sp500_gemini_df
    .sort_values("id_num")
    .drop(columns="id_num")
    .reset_index(drop=True)
)

# ============================================================
# 9) 1-1500 ID KONTROLÜ
# ============================================================

expected_ids = {
    f"SP500_ANN_{i:04d}"
    for i in range(1, 1501)
}

actual_ids = set(
    sp500_gemini_df["annotation_id"]
)

missing_ids = sorted(
    expected_ids - actual_ids
)

extra_ids = sorted(
    actual_ids - expected_ids
)

print("\n======================================")
print("BEKLENEN ID       :", len(expected_ids))
print("MEVCUT UNIQUE ID  :", len(actual_ids))
print("EKSİK ID           :", len(missing_ids))
print("FAZLA ID           :", len(extra_ids))
print("======================================")

if missing_ids:
    print("\nEksik ID'ler:")
    print(missing_ids)

# ============================================================
# 10) GEMINI LABEL KONTROL
# ============================================================

print("\nGemini label dağılımı:")
print(
    sp500_gemini_df["gemini_label"]
    .value_counts(dropna=False)
)

# ============================================================
# 11) KAYDET
# ============================================================

output_file = sp500_dir / output_name

sp500_gemini_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n======================================")
print("KAYDEDİLDİ:")
print(output_file)
print("FINAL SHAPE:", sp500_gemini_df.shape)
print("======================================")

display(sp500_gemini_df.head(10))

CSV  : gemini_labeled_batch_001.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_002.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_003.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_004.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_005.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_006.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_007.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_008.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_009.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_010.csv                  ->   10 kayıt
CSV  : gemini_labeled_batch_011_020_all_100.csv      ->  100 kayıt
CSV  : gemini_labeled_batch_021_040_all_200.csv      ->  200 kayıt
CSV  : gemini_labeled_batch_041_050_all_100.csv      ->  100 kayıt
CSV  : gemini_labeled_batch_051_060_all_100.csv      ->  100 kayıt
CSV  : gemini_labeled_batch_061_080_all_200.csv      ->  200 k

,annotation_id,sample_id,date,finbert_label,finbert_confidence,TEXT,gemini_label,source_file
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03 00:00:00,positive,0.8616266846656799,"U.S. Stocks Higher After Economic Data, Monsan...",positive,gemini_labeled_batch_001.csv
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15 00:00:00,positive,0.8810731172561646,Stock Market Outlook 2024: Rare Bullish Signal...,positive,gemini_labeled_batch_001.csv
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20 00:00:00,negative,0.6254335641860962,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,positive,gemini_labeled_batch_001.csv
3,SP500_ANN_0004,SP500_HEAD_017167,2024-01-04 00:00:00,negative,0.9493247866630554,"US Stock Market Closing: Dow Jones, S&P 500 Re...",negative,gemini_labeled_batch_001.csv
4,SP500_ANN_0005,SP500_HEAD_007040,2019-09-18 00:00:00,negative,0.9478024244308472,"Deutsche Bank: S&P 500 13% Overvalued, Recessi...",negative,gemini_labeled_batch_001.csv
5,SP500_ANN_0006,SP500_HEAD_017097,2023-12-29 00:00:00,neutral,0.8902925848960876,Wall Street strategists’ bull and bear scenari...,neutral,gemini_labeled_batch_001.csv
6,SP500_ANN_0007,SP500_HEAD_000654,2011-07-08 00:00:00,positive,0.375736266374588,European stocks stable before US jobs data,neutral,gemini_labeled_batch_001.csv
7,SP500_ANN_0008,SP500_HEAD_005743,2018-02-26 00:00:00,neutral,0.9330571293830872,"Business News, Strategy, Finance and Corporate...",neutral,gemini_labeled_batch_001.csv
8,SP500_ANN_0009,SP500_HEAD_014540,2023-07-03 00:00:00,positive,0.809818685054779,S&P 500 to Finish Year Modestly Higher: Invesc...,positive,gemini_labeled_batch_001.csv
9,SP500_ANN_0010,SP500_HEAD_009121,2021-03-31 00:00:00,negative,0.6620614528656006,Low Oil Demand Weighing On Helmerich & Payne S...,negative,gemini_labeled_batch_001.csv


In [8]:
# Beklenen 1-1500 ID'ler
expected_ids = {
    f"SP500_ANN_{i:04d}"
    for i in range(1, 1501)
}

# Elimizdeki ID'ler
actual_ids = set(
    sp500_gemini_df["annotation_id"].dropna()
)

# Eksikler
missing_ids = sorted(
    expected_ids - actual_ids
)

print("Eksik ID sayısı:", len(missing_ids))

for x in missing_ids:
    print(x)

Eksik ID sayısı: 40
SP500_ANN_1401
SP500_ANN_1402
SP500_ANN_1403
SP500_ANN_1404
SP500_ANN_1405
SP500_ANN_1406
SP500_ANN_1407
SP500_ANN_1408
SP500_ANN_1409
SP500_ANN_1410
SP500_ANN_1411
SP500_ANN_1412
SP500_ANN_1413
SP500_ANN_1414
SP500_ANN_1415
SP500_ANN_1416
SP500_ANN_1417
SP500_ANN_1418
SP500_ANN_1419
SP500_ANN_1420
SP500_ANN_1421
SP500_ANN_1422
SP500_ANN_1423
SP500_ANN_1424
SP500_ANN_1425
SP500_ANN_1426
SP500_ANN_1427
SP500_ANN_1428
SP500_ANN_1429
SP500_ANN_1430
SP500_ANN_1431
SP500_ANN_1432
SP500_ANN_1433
SP500_ANN_1434
SP500_ANN_1435
SP500_ANN_1436
SP500_ANN_1437
SP500_ANN_1438
SP500_ANN_1439
SP500_ANN_1440


In [9]:
from pathlib import Path
import pandas as pd

# ============================================================
# SP500 HUMAN REVIEW KLASÖRÜ
# ============================================================

human_dir = Path(
    r"D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis"
    r"\db\annotations\sp500_human_review_batches"
)

# ============================================================
# TÜM EXCEL DOSYALARINI BUL
# ============================================================

excel_files = sorted(
    list(human_dir.glob("*.xlsx")) +
    list(human_dir.glob("*.xls"))
)

print("Excel dosya sayısı:", len(excel_files))

for file in excel_files:
    print(file.name)

# ============================================================
# HER DOSYAYI OKU
# ============================================================

human_dfs = []

for file in excel_files:

    try:
        df = pd.read_excel(
            file,
            dtype=str
        )

        # Kaynak dosya bilgisi
        df["source_file"] = file.name

        human_dfs.append(df)

        print(
            f"{file.name:50} -> "
            f"{len(df):4} kayıt | "
            f"{len(df.columns):2} kolon"
        )

    except Exception as e:
        print(
            f"HATA: {file.name} -> {e}"
        )

# ============================================================
# HEPSİNİ TEK DATAFRAME'DE BİRLEŞTİR
# ============================================================

sp500_human_review_df = pd.concat(
    human_dfs,
    ignore_index=True,
    sort=False
)

print("\n======================================")
print("TOPLAM KAYIT:", len(sp500_human_review_df))
print("TOPLAM KOLON:", len(sp500_human_review_df.columns))
print("======================================")

print("\nKolonlar:")
print(sp500_human_review_df.columns.tolist())

display(
    sp500_human_review_df.head(10)
)

Excel dosya sayısı: 106
SP500_annotation_batch_001.xlsx
SP500_annotation_batch_002.xlsx
SP500_annotation_batch_004.xlsx
SP500_annotation_batch_005.xlsx
SP500_annotation_batch_006.xlsx
SP500_annotation_batch_007.xlsx
SP500_annotation_batch_008.xlsx
SP500_annotation_batch_009.xlsx
SP500_annotation_batch_010.xlsx
SP500_annotation_batch_011.xlsx
SP500_annotation_batch_012.xlsx
SP500_annotation_batch_013.xlsx
SP500_annotation_batch_014.xlsx
SP500_annotation_batch_015.xlsx
SP500_annotation_batch_016.xlsx
SP500_annotation_batch_017.xlsx
SP500_annotation_batch_018.xlsx
SP500_annotation_batch_019.xlsx
SP500_annotation_batch_020.xlsx
SP500_annotation_batch_021.xlsx
SP500_annotation_batch_022.xlsx
SP500_annotation_batch_023.xlsx
SP500_annotation_batch_024.xlsx
SP500_annotation_batch_025.xlsx
SP500_annotation_batch_026.xlsx
SP500_annotation_batch_027.xlsx
SP500_annotation_batch_029.xlsx
SP500_annotation_batch_030.xlsx
SP500_annotation_batch_031.xlsx
SP500_annotation_batch_032.xlsx
SP500_annotation

,annotation_id,sample_id,date,text_en,text_tr,finbert_label,finbert_confidence,chatgpt_label,chatgpt_confidence,chatgpt_reason_tr,...,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,SP500 Annotation Batch 107,SP500 Annotation Batch 109
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03,"U.S. Stocks Higher After Economic Data, Monsan...",ABD hisseleri ekonomik veriler ve Monsanto gör...,positive,0.8616266846656799,positive,high,ABD hisseleri yükseliyor; piyasa açısından olu...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15,Stock Market Outlook 2024: Rare Bullish Signal...,2024 borsa görünümü: Nadir bir boğa sinyali S&...,positive,0.8810731172561646,positive,high,Bullish sinyal ve S&P 500'de güçlü yükseliş be...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,Banka krizi korkuları azalırken Fed faiz artır...,negative,0.6254335641860962,positive,medium,Başlık karışık olsa da banka krizi korkularını...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SP500_ANN_0004,SP500_HEAD_017167,2024-01-04,"US Stock Market Closing: Dow Jones, S&P 500 Re...",ABD borsa kapanışı: Fed tutanakları değerlendi...,negative,0.9493247866630554,negative,high,"Dow ve S&P 500 geriliyor, Nasdaq sert düşüyor;...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SP500_ANN_0005,SP500_HEAD_007040,2019-09-18,"Deutsche Bank: S&P 500 13% Overvalued, Recessi...","Deutsche Bank: S&P 500 %13 aşırı değerli, rese...",negative,0.9478024244308472,negative,high,Aşırı değerleme ve resesyon beklentisi açık ne...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SP500_ANN_0006,SP500_HEAD_017097,2023-12-29,Wall Street strategists’ bull and bear scenari...,Wall Street stratejistlerinin 2024 için boğa v...,neutral,0.8902925848960876,neutral,high,Hem olumlu hem olumsuz senaryolardan bahsediyo...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SP500_ANN_0007,SP500_HEAD_000654,2011-07-08,European stocks stable before US jobs data,Avrupa hisseleri ABD istihdam verisi öncesinde...,positive,0.375736266374588,neutral,high,Hisseler stabil/dengede; veri öncesi net oluml...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SP500_ANN_0008,SP500_HEAD_005743,2018-02-26,"Business News, Strategy, Finance and Corporate...","İş dünyası haberleri, strateji, finans ve kuru...",neutral,0.9330571293830872,neutral,high,Genel kategori/başlık gibi; net finansal duygu...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,SP500_ANN_0009,SP500_HEAD_014540,2023-07-03,S&P 500 to Finish Year Modestly Higher: Invesc...,Invesco'dan Hooper'a göre S&P 500 yılı mütevaz...,positive,0.809818685054779,positive,high,S&P 500 için yıl sonuna doğru yükseliş beklent...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SP500_ANN_0010,SP500_HEAD_009121,2021-03-31,Low Oil Demand Weighing On Helmerich & Payne S...,Düşük petrol talebi Helmerich & Payne hissesi ...,negative,0.6620614528656006,negative,high,Düşük talep hisse üzerinde baskı yaratıyor; aç...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# annotation_id bazında duplicate varsa önce temizle
sp500_human_review_df = (
    sp500_human_review_df
    .drop_duplicates(subset="annotation_id", keep="first")
    .reset_index(drop=True)
)

sp500_gemini_df = (
    sp500_gemini_df
    .drop_duplicates(subset="annotation_id", keep="first")
    .reset_index(drop=True)
)

# annotation_id üzerinden birleştir
sp500_master_df = sp500_human_review_df.merge(
    sp500_gemini_df,
    on="annotation_id",
    how="left",
    suffixes=("_human", "_gemini")
)

print("Human review kayıt:", len(sp500_human_review_df))
print("Gemini kayıt:", len(sp500_gemini_df))
print("Master kayıt:", len(sp500_master_df))

print("\nKolonlar:")
print(sp500_master_df.columns.tolist())

display(sp500_master_df.head(10))

Human review kayıt: 1031
Gemini kayıt: 1460
Master kayıt: 1031

Kolonlar:
['annotation_id', 'sample_id_human', 'date_human', 'text_en', 'text_tr', 'finbert_label_human', 'finbert_confidence_human', 'chatgpt_label', 'chatgpt_confidence', 'chatgpt_reason_tr', 'finbert_correctness', 'finbert_correctness_note', 'Unnamed: 12', 'Özet', 'Değer', 'source_file_human', 'final_label', 'Unnamed: 13', 'Unnamed: 14', 'ANNOTATION BATCH 102', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'SP500 Annotation Batch 107', 'SP500 Annotation Batch 109', 'sample_id_gemini', 'date_gemini', 'finbert_label_gemini', 'finbert_confidence_gemini', 'TEXT', 'gemini_label', 'source_file_gemini']


,annotation_id,sample_id_human,date_human,text_en,text_tr,finbert_label_human,finbert_confidence_human,chatgpt_label,chatgpt_confidence,chatgpt_reason_tr,...,Unnamed: 11,SP500 Annotation Batch 107,SP500 Annotation Batch 109,sample_id_gemini,date_gemini,finbert_label_gemini,finbert_confidence_gemini,TEXT,gemini_label,source_file_gemini
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03,"U.S. Stocks Higher After Economic Data, Monsan...",ABD hisseleri ekonomik veriler ve Monsanto gör...,positive,0.8616266846656799,positive,high,ABD hisseleri yükseliyor; piyasa açısından olu...,...,NaN,NaN,NaN,SP500_HEAD_000004,2008-01-03 00:00:00,positive,0.8616266846656799,"U.S. Stocks Higher After Economic Data, Monsan...",positive,gemini_labeled_batch_001.csv
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15,Stock Market Outlook 2024: Rare Bullish Signal...,2024 borsa görünümü: Nadir bir boğa sinyali S&...,positive,0.8810731172561646,positive,high,Bullish sinyal ve S&P 500'de güçlü yükseliş be...,...,NaN,NaN,NaN,SP500_HEAD_016918,2023-12-15 00:00:00,positive,0.8810731172561646,Stock Market Outlook 2024: Rare Bullish Signal...,positive,gemini_labeled_batch_001.csv
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,Banka krizi korkuları azalırken Fed faiz artır...,negative,0.6254335641860962,positive,medium,Başlık karışık olsa da banka krizi korkularını...,...,NaN,NaN,NaN,SP500_HEAD_013289,2023-03-20 00:00:00,negative,0.6254335641860962,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,positive,gemini_labeled_batch_001.csv
3,SP500_ANN_0004,SP500_HEAD_017167,2024-01-04,"US Stock Market Closing: Dow Jones, S&P 500 Re...",ABD borsa kapanışı: Fed tutanakları değerlendi...,negative,0.9493247866630554,negative,high,"Dow ve S&P 500 geriliyor, Nasdaq sert düşüyor;...",...,NaN,NaN,NaN,SP500_HEAD_017167,2024-01-04 00:00:00,negative,0.9493247866630554,"US Stock Market Closing: Dow Jones, S&P 500 Re...",negative,gemini_labeled_batch_001.csv
4,SP500_ANN_0005,SP500_HEAD_007040,2019-09-18,"Deutsche Bank: S&P 500 13% Overvalued, Recessi...","Deutsche Bank: S&P 500 %13 aşırı değerli, rese...",negative,0.9478024244308472,negative,high,Aşırı değerleme ve resesyon beklentisi açık ne...,...,NaN,NaN,NaN,SP500_HEAD_007040,2019-09-18 00:00:00,negative,0.9478024244308472,"Deutsche Bank: S&P 500 13% Overvalued, Recessi...",negative,gemini_labeled_batch_001.csv
5,SP500_ANN_0006,SP500_HEAD_017097,2023-12-29,Wall Street strategists’ bull and bear scenari...,Wall Street stratejistlerinin 2024 için boğa v...,neutral,0.8902925848960876,neutral,high,Hem olumlu hem olumsuz senaryolardan bahsediyo...,...,NaN,NaN,NaN,SP500_HEAD_017097,2023-12-29 00:00:00,neutral,0.8902925848960876,Wall Street strategists’ bull and bear scenari...,neutral,gemini_labeled_batch_001.csv
6,SP500_ANN_0007,SP500_HEAD_000654,2011-07-08,European stocks stable before US jobs data,Avrupa hisseleri ABD istihdam verisi öncesinde...,positive,0.375736266374588,neutral,high,Hisseler stabil/dengede; veri öncesi net oluml...,...,NaN,NaN,NaN,SP500_HEAD_000654,2011-07-08 00:00:00,positive,0.375736266374588,European stocks stable before US jobs data,neutral,gemini_labeled_batch_001.csv
7,SP500_ANN_0008,SP500_HEAD_005743,2018-02-26,"Business News, Strategy, Finance and Corporate...","İş dünyası haberleri, strateji, finans ve kuru...",neutral,0.9330571293830872,neutral,high,Genel kategori/başlık gibi; net finansal duygu...,...,NaN,NaN,NaN,SP500_HEAD_005743,2018-02-26 00:00:00,neutral,0.9330571293830872,"Business News, Strategy, Finance and Corporate...",neutral,gemini_labeled_batch_001.csv
8,SP500_ANN_0009,SP500_HEAD_014540,2023-07-03,S&P 500 to Finish Year Modestly Higher: Invesc...,Invesco'dan Hooper'a göre S&P 500 yılı mütevaz...,positive,0.809818685054779,positive,high,S&P 500 için yıl sonuna doğru yükseliş beklent...,...,NaN,NaN,NaN,SP500_HEAD_014540,2023-07-03 00:00:00,positive,0.809818685054779,S&P 500 to Finish Year Modestly Higher: Invesc...,positive,gemini_labeled_batch_001

In [12]:
# Kaç farklı annotation_id var?
print(
    "Farklı annotation_id:",
    sp500_master_df["annotation_id"].nunique()
)

# Gemini ve ChatGPT etiketleri aynı mı?
same = (
    sp500_master_df["gemini_label"] ==
    sp500_master_df["chatgpt_label"]
)

print("Toplam kayıt:", len(sp500_master_df))
print("Aynı etiket:", same.sum())
print("Farklı etiket:", (~same).sum())
print(
    "Uyum oranı: %.2f%%" %
    (same.mean() * 100)
)

Farklı annotation_id: 1030
Toplam kayıt: 1031
Aynı etiket: 944
Farklı etiket: 87
Uyum oranı: 91.56%


In [13]:
# Kaç farklı annotation_id var?
print(
    "Farklı annotation_id:",
    sp500_master_df["annotation_id"].nunique()
)

# Gemini ve ChatGPT etiketleri aynı mı?
same = (
    sp500_master_df["gemini_label"] ==
    sp500_master_df["chatgpt_label"]
)

print("Toplam kayıt:", len(sp500_master_df))
print("Aynı etiket:", same.sum())
print("Farklı etiket:", (~same).sum())
print(
    "Uyum oranı: %.2f%%" %
    (same.mean() * 100)
)

Farklı annotation_id: 1030
Toplam kayıt: 1031
Aynı etiket: 944
Farklı etiket: 87
Uyum oranı: 91.56%
